# Curve-Supervised Hard3 - Google Colab
Once `pseudo_smoke` ile boru hattini dogrulayin. Yayin deneyi icin gercek hairline/jaw curve manifesti gerekir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
print(CODE_ROOT)

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/orthodontic/data/dataset')
MANIFEST = Path('/content/drive/MyDrive/orthodontic/annotations/hard3_curves_v1.json')
assert DATA_ROOT.exists(), f'Dataset bulunamadi: {DATA_ROOT}'
print('Dataset:', DATA_ROOT)
print('Manifest exists:', MANIFEST.exists())

## 1. Hizli pseudo-curve smoke testi

In [ ]:
%cd /content/comparative-study
!python -u curve_supervised_hard3_refinement/colab_run_curve_hard3.py --preset pseudo_smoke --seed 42

## 2. Bos anotasyon manifesti
Bu hucre mevcut dolu manifestin uzerine yazmamasi icin yalniz dosya yoksa calisir.

In [ ]:
if not MANIFEST.exists():
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'python', '-u',
        str(CODE_ROOT / 'curve_supervised_hard3_refinement/prepare_annotations.py'),
        '--data-root', str(DATA_ROOT),
        '--output', str(MANIFEST),
    ], check=True, cwd=str(CODE_ROOT))
else:
    print('Mevcut manifest korundu:', MANIFEST)

## 3. Gercek curve anotasyonlu Fold-1 deneyi
Manifestte dis-train icinde en az 60 tam anotasyon olmadan bu hucre bilincli olarak durur.

In [ ]:
!python -u curve_supervised_hard3_refinement/colab_run_curve_hard3.py --preset annotated_fold1 --seed 42 --annotation-manifest {MANIFEST} --minimum-annotated-samples 60